In [9]:
!pip install tqdm

ERROR! Session/line number was not unique in database. History logging moved to new session 620
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [4]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        """
        idx -> (B,T)
        targets -> (B,T)
        """
        logits = self.token_embedding_table(idx)  # (B,T,C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # (B,C)
            probs = F.softmax(logits, dim=-1)  # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx



In [5]:

class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


In [27]:
text = ""
with open("data/shakespeare.txt", "r") as f:
    text=f.read()

assert len(text) != 0

print(len(text))
text = text[0: len(text)//200]
print(len(text))


1115389
5576


In [28]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
data = [stoi[c] for c in text]

block_size = 8
dataset = TextDataset(data, block_size)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in tqdm(range(200)):
    for x, y in dataloader:
        logits, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")




  0%|          | 1/200 [00:00<01:38,  2.03it/s]

Epoch 0 Loss: 2.6272


 10%|█         | 21/200 [00:09<01:23,  2.14it/s]

Epoch 20 Loss: 2.3759


 20%|██        | 41/200 [00:19<01:15,  2.12it/s]

...

 80%|████████  | 161/200 [01:15<00:18,  2.15it/s]

Epoch 160 Loss: 2.2322


 90%|█████████ | 181/200 [01:24<00:08,  2.19it/s]

Epoch 180 Loss: 2.5099


100%|██████████| 200/200 [01:33<00:00,  2.14it/s]


In [29]:
start_idx = torch.tensor([[stoi['h']]], dtype=torch.long)  # start from 'h'
generated_idx = model.generate(start_idx, max_new_tokens=1000)
generated_text = ''.join([itos[i.item()] for i in generated_idx[0]])
print("Generated text:", generated_text)

Generated text: harconk-
ven u h ode smpr ally aryoutie ifrtos istheneay You.
e pacir r y d condelamorcon kes y tuistak--


Thecuncaru, ompene ocadouchoke:
Thatot, y ag as we hatoureir asthe wisotrs tshealir ank'sthin?
in brellimy, an t Canemay o we ouse whongocthbecoore ccown
Thesprmuf m.
ilonerongu, bjer sthey-hensgemey
MENIUSothe un the Yo aicononot o m t is atheleal th fous congoureccle coghevertthoneyou at, t: s ou be.
S:
Yoven:


she ando
Lin?
MENI

S: pant ckees eenangey felit co

Ale virve beeanoust trngar it sprne who om
icomid
Hacordeal t endelis wen: tit s whe u s enonopire be


f arepelense y w fo.
Ben fous Citode.
Se whowher Rot; mou Cane ud p, rs, titat w s o, tod; wous, wely ker cir, he MENENI's 'se omaien pesurstok----
ise the.
An
Dir st te undsir, 'e-
stheeconsue wath cthean is, at ven neles



Ag ak. murt:
Vey! nkick as, wherenenl orspe, hid tizealis y, t amall thirs Wize'danneyobende r:

A yo

Yo frealeawaly he I

h?
acite me antry on Cain Cod d hatha cidizedod ty we